# Parameters

In [9]:
class args():
    def __init__(self):
        self.DEVICE = "cuda"
        self.EPOCHS = 10
        self.GAMMA = 0.1
        self.LEARNING_RATE = 1e-4
        self.SEED = 42
        self.STEPS = 5
        self.TRAINING_BATCH_SIZE = 8
        self.TESTING_BATCH_SIZE = 1

# Data

In [10]:
import cv2
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import Compose, Normalize, Resize, ToTensor

class Paw(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

        self.path_train = "../input/petfinder-pawpularity-score/train/"

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        # image features
        image_id = self.dataframe["Id"][idx]
        image_path = os.path.join(self.path_train, image_id + ".jpg")
        image = cv2.imread(image_path)
        image = self.transform(image)

        # other features
        focus = self.dataframe["Subject Focus"][idx]
        eyes = self.dataframe["Eyes"][idx]
        face = self.dataframe["Face"][idx]
        near = self.dataframe["Near"][idx]
        action = self.dataframe["Action"][idx]
        accessory = self.dataframe["Accessory"][idx]
        group = self.dataframe["Group"][idx]
        collage = self.dataframe["Collage"][idx]
        human = self.dataframe["Human"][idx]
        occlusion = self.dataframe["Occlusion"][idx]
        info = self.dataframe["Info"][idx]
        blur = self.dataframe["Blur"][idx]
        all_features = [focus, eyes, face, near, action, accessory, group, collage, human, occlusion, info, blur]
        features = np.array(all_features).astype(np.float32)

        # label
        pawpularity = self.dataframe["Pawpularity"][idx]
        label = np.array(pawpularity).astype(np.int64)

        return image, features, label

class Data():
    def __init__(self, cfg):
        self.cfg = cfg

    def build_transforms(self):
        self.training_transforms = Compose([
            ToTensor(),
            Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225]),
            Resize((224, 224))
        ])
        self.testing_transforms = Compose([
            ToTensor(),
            Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225]),
            Resize((224, 224))
        ])

    def build_data(self):
        dataframe = pd.read_csv("../input/petfinder-pawpularity-score/train.csv")
        train_df, val_df = train_test_split(dataframe, test_size=0.2, random_state=self.cfg.SEED)
        self.training_data = Paw(train_df, transform=self.training_transforms)
        self.testing_data = Paw(val_df, transform=self.testing_transforms)
    def build_dataloader(self):
        self.training_dataloader = DataLoader(
            self.training_data, batch_size=self.cfg.TRAINING_BATCH_SIZE, shuffle=True)
        self.testing_dataloader = DataLoader(
            self.testing_data, batch_size=self.cfg.TESTING_BATCH_SIZE, shuffle=False)


# Model

In [11]:
import torch
from torch import nn
from torchvision.models import resnet50
from torchvision.models import ResNet50_Weights

class PawNet(nn.Module):
    def __init__(self):
        super(PawNet, self).__init__()
        self.resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        del(self.resnet.classifier)
        features = list(self.resnet.features)
        self.layers = nn.ModuleList(features).eval()

    def forward(self, image, features):
        image = self.resnet(image)
        features = self.linear(features)
        return image + features

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\Alan/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:01<00:00, 54.5MB/s]
